# 🚛 Outil de Planification Transport — Auchan Région Nord
### Guide complet à destination des nouveaux utilisateurs

---

> **Version** : v7 · **Périmètre** : Région Nord (Lesquin, Douai, …) · **Technologie** : application web 100 % navigateur, aucune installation requise

---

## Table des matières

1. [Vue d'ensemble](#1-vue-densemble)
2. [Prérequis et fichiers d'entrée](#2-prérequis-et-fichiers-dentrée)
3. [Structure du fichier Excel principal](#3-structure-du-fichier-excel-principal)
4. [Format du distancier](#4-format-du-distancier)
5. [Format de la cartographie réception](#5-format-de-la-cartographie-réception)
6. [Interface utilisateur](#6-interface-utilisateur)
7. [Workflow pas à pas](#7-workflow-pas-à-pas)
8. [Concepts clés](#8-concepts-clés)
9. [Types de marchandises et matrice de compatibilité](#9-types-de-marchandises-et-matrice-de-compatibilité)
10. [Règles métier](#10-règles-métier)
11. [Moteur d'optimisation automatique](#11-moteur-doptimisation-automatique)
12. [Tableau de bord KPI](#12-tableau-de-bord-kpi)
13. [Gestion des sessions](#13-gestion-des-sessions)
14. [Astuces et raccourcis](#14-astuces-et-raccourcis)

---
## 1. Vue d'ensemble

L'**Outil de Planification Transport v7** est une application web autonome (fichier `.html`) qui permet de :

- **Visualiser** les flux de livraison à planifier (magasin, volume, type de marchandise, jour)
- **Construire et optimiser** les tournées de transport (quel camion livre quels magasins, dans quel ordre, à quelle heure)
- **Organiser** les tournées en **modules chauffeur** (enchaînement de tournées dans la journée d'un conducteur)
- **Valider** le respect des règles métier (capacité, compatibilité marchandises, temps de conduite, créneaux magasin…)
- **Exporter** le planning en JSON (sauvegarde) ou CSV (pour transmission)

### Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                     Navigateur web                          │
│                                                             │
│  ┌───────────┐   ┌───────────────┐   ┌──────────────────┐  │
│  │ Fichier   │   │  Distancier   │   │  Cartographie    │  │
│  │  Excel    │   │  (km + min)   │   │  réception CSV   │  │
│  └─────┬─────┘   └──────┬────────┘   └────────┬─────────┘  │
│        └────────────────┴────────────────────▼             │
│                    outil_v7.html                            │
│               (100 % client-side, JS pur)                   │
│                                                             │
│  ┌──────────┐  ┌──────────────────────┐  ┌──────────────┐  │
│  │ Panneau  │  │    Gantt modules /   │  │  Panel       │  │
│  │  Flux    │  │  remorques / magasins│  │  tournées    │  │
│  └──────────┘  └──────────────────────┘  └──────────────┘  │
└─────────────────────────────────────────────────────────────┘
```

> **Aucun serveur, aucune installation.** Toutes les données restent dans le navigateur. L'application fonctionne hors ligne une fois le fichier ouvert.

---
## 2. Prérequis et fichiers d'entrée

### 2.1 Prérequis techniques

| Élément | Requis |
|---|---|
| Navigateur | Chrome ou Edge (recommandé), Firefox |
| Connexion internet | Uniquement au 1er chargement (bibliothèque SheetJS) |
| Serveur | ❌ Aucun |
| Installation | ❌ Aucune |

**Pour ouvrir l'outil** : double-cliquer sur `outil_v7.html`.

---

### 2.2 Fichiers d'entrée

| Fichier | Format | Obligatoire | Rôle |
|---|---|---|---|
| Fichier de données principal | `.xlsx` / `.xls` | ✅ Oui | Flux, tournées, lieux, plages d'ouverture |
| Distancier Fleet | `.xlsx` / `.xls` | ✅ Oui | Distances (km) et temps (min) entre chaque paire entrepôt ↔ magasin |
| Cartographie réception | `.csv` | ⚠️ Recommandé | Créneaux horaires de réception par magasin et par jour |

> **Persistance du distancier** : une fois chargé, le distancier est mémorisé dans le `localStorage` du navigateur. Il n'est pas nécessaire de le recharger à chaque session.

---
## 3. Structure du fichier Excel principal

Le fichier Excel contient **plusieurs onglets**. Voici les onglets lus par l'application :

### Onglet `Flux`

Chaque ligne représente un flux de livraison à planifier.

| Colonne | Nom | Description | Exemple |
|---|---|---|---|
| A | Entrepôt | Lieu de départ du camion | `LESQUIN` |
| B | Magasin | Magasin de destination | `AUCHAN RONCQ` |
| C | Marchandise | Type de produit | `PGC` |
| D | UT | Volume en Unités de Transport | `12` |
| E | Jour | Jour de livraison | `Lundi` |

### Onglet `Tournées`

Définit les tournées disponibles (créneaux horaires, capacités).

| Colonne | Nom | Description | Exemple |
|---|---|---|---|
| A | ID Tournée | Identifiant unique | `T01`, `T02` |
| B | Capacité | Capacité en UT | `33` |
| C | Retour | Entrepôt de retour | `LESQUIN` |
| D | Accrochage | Durée d'accrochage remorque (min) | `15` |
| E | Chargement | Durée de chargement tracteur (min) | `30` |
| F | Heure début | Heure de départ | `06:00` |
| G | Entrepôt | Entrepôt de départ | `LESQUIN` |
| H | Jour | Jour d'exploitation | `Lundi` |

### Onglet `Lieux`

Référentiel des lieux (entrepôts et magasins).

| Colonne | Nom | Description |
|---|---|---|
| A | Nom | Nom du lieu |
| B | Type | `Entrepôt` ou `Magasin` |
| C | Zone logistique | Zone de rattachement (ex. `Nord`, `Est`) |
| D | Zone livraison | Zone géographique de livraison |

### Onglet `PlagesOuverture` *(optionnel)*

Créneaux d'ouverture des magasins (complément à la cartographie CSV).

| Colonne | Description |
|---|---|
| A | Nom du magasin |
| B | Jour |
| C | Heure d'ouverture (`HH:MM`) |
| D | Heure de fermeture (`HH:MM`) |
| E | Type de marchandise concerné (optionnel) |

---
## 4. Format du distancier

Le distancier est un fichier Excel avec **une ligne par paire de lieux** :

| Colonne | Nom | Description | Exemple |
|---|---|---|---|
| De | Lieu de départ | Nom exact du lieu | `LESQUIN` |
| Vers | Lieu d'arrivée | Nom exact du lieu | `AUCHAN RONCQ` |
| Km | Distance | En kilomètres | `18.4` |
| Minutes | Durée | Temps de trajet en minutes | `25` |

> **Important** : les noms de lieux doivent correspondre **exactement** aux noms utilisés dans le fichier Excel des flux. Les distances sont **bidirectionnelles** — si `A → B` est renseigné, `B → A` n'est pas obligatoire mais recommandé.

### Cas particulier ENT IENA

L'entrepôt `ENT IENA` dans le distancier est automatiquement dupliqué pour correspondre à `ENT IENA1` et `ENT IENA2` dans les flux. Il suffit de renseigner une seule ligne `ENT IENA` dans le distancier.

### Alerte distances manquantes

Si une paire entrepôt ↔ magasin est absente du distancier, l'application affiche une **alerte orange** en haut du Gantt : les trajets concernés auront une durée affichée à **0 minute**, ce qui faussera le planning. Pensez à compléter le distancier.


---
## 5. Format de la cartographie réception

La cartographie est un fichier **CSV** (séparateur `;`) qui précise les créneaux de réception de chaque magasin.

### Structure des colonnes

```
Magasin ; NAL_Lun ; NAL_Mar ; … ; NAL_Sam ; Frais_Lun ; … ; Frais_Sam ; Surg_Lun ; … ; Surg_Sam
```

Soit **19 colonnes** : 1 nom + 6 jours × 3 catégories (SEC/Frais/Surgelés).

### Format des créneaux

| Format | Signification |
|---|---|
| `07:00 - 12:00` | Créneau unique de 7h à 12h |
| `07:00 - 10:00 / 14:00 - 17:00` | Deux créneaux dans la journée |
| `Fermé` | Magasin fermé ce jour |
| *(vide)* | Pas de contrainte renseignée |

> **Créneaux nocturnes** : si un créneau se termine après minuit (ex. `22:00 - 06:00`), l'application le gère automatiquement en ajoutant 24h au curseur.

---
## 6. Interface utilisateur

### 6.1 Barre d'en-tête

```
┌──────────────────────────────────────────────────────────────────────────────┐
│ 🚛 Outil Planification Transport  Auchan — Région Nord                       │
│                    [📂 Charger Excel] [📏 Distancier] [📋 Cartographie]      │
│                    [🔄 Actualiser] [📥 Importer] [💾 Exporter JSON]          │
│                    [📋 CSV] [🔍 Valider]                                      │
└──────────────────────────────────────────────────────────────────────────────┘
```

| Bouton | Rôle |
|---|---|
| 📂 Charger Excel | Charge le fichier de données principal |
| 📏 Distancier | Charge le fichier de distances (mémorisé en local) |
| 📋 Cartographie | Charge les créneaux de réception magasins (CSV) |
| 🔄 Actualiser | Relit les données sans recharger la page |
| 📥 Importer | Restaure une session précédemment exportée (JSON) |
| 💾 Exporter JSON | Sauvegarde le plan complet en fichier JSON |
| 📋 CSV | Exporte le planning en tableur CSV |
| 🔍 Valider | Ouvre le panneau de validation des règles métier |

---

### 6.2 Barre d'outils (toolbar)

#### Filtres (partie gauche)

| Filtre | Description |
|---|---|
| **Jour** | Sélectionne le jour affiché (Lundi … Dimanche) |
| **Marchandise** | Filtre multi-sélection : PGC, NAL, BSA, FL, PF, SURG |
| **Magasin** | Filtre par magasin |
| **Entrepôt** | Filtre par entrepôt de départ |
| **Zone log.** | Filtre par zone logistique |
| **Zone liv.** | Filtre par zone de livraison |
| ↔ Navette | Crée une navette entre deux entrepôts |

#### Actions (partie droite)

| Bouton | Rôle |
|---|---|
| 📦 Modules | Vue Gantt par modules chauffeur |
| 🚛 Remorques | Vue Gantt par remorques physiques |
| 🏪 Magasins | Vue Gantt par magasin |
| − / + | Dézoomer / Zoomer la timeline |
| 🧠 Optimiser | Lance l'optimisation automatique (heuristique + LNS) |
| ♻️ Reset | Réinitialise tout le plan |
| + Module | Ajoute un module chauffeur manuellement |
| + Tournée | Ajoute une tournée au module sélectionné |

---

### 6.3 Disposition principale (3 colonnes)

```
┌──────────────┬─────────────────────────────────┬──────────────────┐
│              │                                 │                  │
│  📋 Flux à   │   📦 Gantt — Modules /          │  📋 Paramètres   │
│  planifier   │   🚛 Remorques / 🏪 Magasins    │  tournées        │
│              │                                 │                  │
│  ─────────── │   ──── Timeline ────────────    │  (clic sur un   │
│  UT total    │   M1: [T01──────][T02──────]    │   module)        │
│  Chargées    │   M2: [T03──────────────────]   │                  │
│  Restantes   │   M3: [T04──][T05──────────]   │                  │
│  Couverture  │                                 │                  │
└──────────────┴─────────────────────────────────┴──────────────────┘
```

#### Panneau gauche — Flux à planifier

- Liste tous les flux du jour sélectionné (selon les filtres actifs)
- Chaque flux indique : magasin, entrepôt, type de marchandise, volume (UT)
- Les flux déjà planifiés apparaissent **estompés** (✅ assignés)
- **Glisser-déposer** : faites glisser un flux sur une barre de tournée dans le Gantt
- En bas : statistiques UT total / chargées / restantes / **taux de couverture**

#### Zone centrale — Gantt

Trois vues accessibles via les boutons de la toolbar :

| Vue | Description |
|---|---|
| **📦 Modules** | Vue principale — un module (chauffeur) par ligne, avec ses tournées chainées |
| **🚛 Remorques** | Vue par remorque physique — utile pour vérifier les conflits d'accès au quai |
| **🏪 Magasins** | Vue par magasin — montre quand et combien de fois un magasin est livré |

#### Panneau droit — Paramètres tournées

Visible en **vue Modules**. Cliquez sur un module pour voir la liste de ses tournées avec :
- Identifiant de la tournée, UT libres, taux de remplissage
- Sélecteur d'entrepôt de **retour**
- Durée d'**accrochage** (connexion remorque)
- Barre de composition de la remorque (couleurs par lieu)
- Liste des flux (magasin, UT, temps de déchargement, type de marchandise, délai emballage)
- Boutons ▲▼ pour réordonner les flux dans la tournée

---
## 7. Workflow pas à pas

### Étape 1 — Charger les données

1. Cliquer sur **📂 Charger Excel** → sélectionner le fichier `.xlsx`
2. Cliquer sur **📏 Distancier** → sélectionner le distancier (une seule fois, mémorisé ensuite)
3. *(Optionnel)* Cliquer sur **📋 Cartographie** → sélectionner le fichier CSV

Une fois chargés, le Gantt s'affiche automatiquement avec les tournées du jour.

---

### Étape 2 — Sélectionner le jour

Utilisez le menu déroulant **Jour** dans la toolbar pour passer d'un jour à l'autre.  
Le Gantt et le panneau flux se mettent à jour instantanément.

---

### Étape 3 — Planifier les flux

#### Option A — Optimisation automatique (recommandée)

Cliquer sur **🧠 Optimiser**.  
L'algorithme effectue en ~30–60 s :
1. **Phase 1 — Construction gloutonne** : affecte chaque flux à la tournée existante la moins coûteuse, ou crée une nouvelle tournée
2. **Phase 2 — LNS** (Large Neighbourhood Search, 500 itérations) : retire et réinsère des groupes de flux pour améliorer la solution
3. **Consolidation** : regroupe les tournées en modules chauffeur valides
4. **Correction anti-chevauchement** : ajuste les heures de départ pour éliminer les superpositions

#### Option B — Planification manuelle

- **Glisser-déposer** un flux depuis le panneau gauche vers la barre d'une tournée
- Si le flux ne tient pas en entier : une fenêtre demande combien d'UT assigner (fraction possible par pas de 0,5 UT)
- Réordonner les flux avec les boutons ▲▼ dans le panneau droit

#### Option C — Hybride

Lancer 🧠 Optimiser, puis ajuster manuellement les tournées problématiques.

---

### Étape 4 — Ajuster les modules

| Action | Comment |
|---|---|
| Déplacer une tournée dans un autre module | Glisser la barre de tournée vers une autre ligne |
| Changer l'heure de départ d'une tournée | Glisser la barre horizontalement |
| Ajouter un module | Bouton **+ Module** |
| Ajouter une tournée à un module | Bouton **+ Tournée** |
| Insérer une coupure de shift (✂) | Cliquer sur la zone de découpe entre deux tournées |
| Déplacer une pause chauffeur | Glisser le bloc ⏸ vers la gauche |

---

### Étape 5 — Valider

Cliquer sur **🔍 Valider**.

- Sélectionner une tournée dans la liste pour voir sa checklist détaillée
- Ou cliquer **Tout valider** pour un récapitulatif de toutes les tournées et modules
- Les règles en ❌ rouge indiquent les non-conformités à corriger

---

### Étape 6 — Exporter

| Bouton | Usage |
|---|---|
| 💾 Exporter JSON | Sauvegarde complète du plan (peut être réimportée avec 📥 Importer) |
| 📋 CSV | Tableau exploitable dans Excel ou transmis à d'autres équipes |

---
## 8. Concepts clés

### 8.1 UT — Unité de Transport

L'**UT** (Unité de Transport) est l'unité de mesure du volume de marchandises dans une remorque.

- **Capacité maximale** d'une remorque standard : **33 UT**
- Chaque flux a un volume en UT (entier ou fraction par 0,5)
- Un flux peut être **fractionné** sur plusieurs tournées si le volume dépasse la place disponible

---

### 8.2 Flux

Un **flux** est une livraison élémentaire :

```
Entrepôt → Magasin  |  Marchandise  |  Volume (UT)  |  Jour
```

Un même magasin peut recevoir plusieurs flux par jour (de types de marchandises différents ou de tournées différentes).

---

### 8.3 Tournée

Une **tournée** est l'itinéraire complet d'un tracteur :

```
Entrepôt → [Accrochage remorque] → [Chargement] → Magasin 1 → Magasin 2 → … → Entrepôt retour
```

| Paramètre | Description | Valeur type |
|---|---|---|
| **Capacité** | Limite en UT | 33 UT |
| **Entrepôt départ** | Lieu de chargement | LESQUIN, DOUAI |
| **Entrepôt retour** | Lieu de fin | LESQUIN, DOUAI |
| **Accrochage** | Temps de connexion remorque | 15 min |
| **Chargement** | Temps de chargement tracteur (frais uniquement) | 30 min |
| **Heure de départ** | Offset depuis minuit en minutes | ex. 360 = 06:00 |

#### Temps de déchargement (barème fixe)

| Volume livré | Durée de déchargement |
|---|---|
| 1 – 9 UT | 15 min |
| 10 – 18 UT | 30 min |
| 19 – 33 UT | 45 min |

---

### 8.4 Module

Un **module** représente la journée de travail d'un chauffeur : plusieurs tournées enchaînées.

```
Module M2 :  [Tournée T03 ─────────][Tournée T07 ──────────────────]
              06:00 → 10:30          11:00 → 16:45
              ←──────────── Shift total : 10h45 ────────────────────►
```

- **M0** (module de construction) : module spécial en haut du Gantt, épinglé en position sticky, utilisé pour construire les tournées avant de les affecter à un module réel
- Les modules M1, M2… correspondent à des chauffeurs/tracteurs réels

---

### 8.5 Shift chauffeur

Le **shift** est la plage de travail continue du chauffeur dans un module (entre deux coupures ✂).

- Représenté par une **barre rouge** (jour) ou **violette** (nuit) dans le Gantt
- Durée maximale légale : **11h en journée**, **10h la nuit** (plage nuit = 22h00–06h00)
- Une **coupure de shift** ✂ peut être insérée manuellement entre deux tournées pour repartir le compteur
- L'optimiseur insère des coupures automatiquement si nécessaire

---

### 8.6 Navette

Une **navette** est un trajet direct entre deux entrepôts (sans magasin livré), utilisé pour transférer des marchandises ou repositionner un tracteur.

---
## 9. Types de marchandises et matrice de compatibilité

### Types de marchandises

| Code | Libellé | Contraintes particulières |
|---|---|---|
| **PGC** | Produits Grande Consommation | Chargement uniquement de **jour** (≥ 06h00) |
| **NAL** | Non-Alimentaire | Chargement uniquement de **jour** (≥ 06h00) |
| **BSA** | Boissons Sans Alcool | Chargement uniquement de **jour** (≥ 06h00) |
| **FL** | Fruits & Légumes 🌙 | Transport nocturne possible ; incompatible avec SEC |
| **PF** | Produits Frais 🌙 | Transport nocturne possible ; incompatible avec SEC |
| **SURG** | Surgelés | Incompatible avec tout autre type |

> 🌙 FL et PF peuvent être chargés avant 6h00 (livraisons de nuit).

---

### Matrice de compatibilité (R2)

Indique si deux types de marchandises peuvent cohabiter dans la **même remorque**.

| | PGC | NAL | BSA | FL | PF | SURG |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| **PGC** | ✅ | ✅ | ✅ | ❌ | ❌ | ❌ |
| **NAL** | ✅ | ✅ | ✅ | ❌ | ❌ | ❌ |
| **BSA** | ✅ | ✅ | ✅ | ❌ | ❌ | ❌ |
| **FL**  | ❌ | ❌ | ❌ | ✅ | ✅ | ❌ |
| **PF**  | ❌ | ❌ | ❌ | ✅ | ✅ | ❌ |
| **SURG**| ❌ | ❌ | ❌ | ❌ | ❌ | ✅ |

> **SEC** (PGC + NAL + BSA) ne peut pas être transporté avec du **frais** (FL + PF) ni des **surgelés**.  
> Les **surgelés** voyagent seuls.

---
## 10. Règles métier

Le panneau **🔍 Valider** contrôle automatiquement les règles suivantes.

### Règles par tournée

| Règle | Titre | Description | Conséquence si violation |
|---|---|---|---|
| **R1** | Capacité remorque | Total UT ≤ 33 UT par tournée | ❌ Tournée invalide |
| **R2** | Compatibilité marchandises | Respect de la matrice frais/sec/surgelé | ❌ Tournée invalide |
| **R3** | Chargement diurne PGC | PGC/NAL/BSA chargés uniquement après 06h00 | ❌ Tournée invalide |
| **R4** | Temps de déchargement | Barème 15/30/45 min selon UT | ℹ️ Informatif |
| **R6** | Pause conduite (4h30) | Pause 45 min insérée automatiquement après 4h30 de conduite | ✅ Automatique |
| **R7** | Pause service (6h) | Pause 30 min supplémentaire après 6h de service | ✅ Automatique |
| **R10** | Créneaux horaires | Livraison dans les plages d'ouverture du magasin | ❌ Hors créneau |
| **R11** | Reprise emballages | Au moins 1 passage/jour/magasin | ℹ️ Informatif |

### Règles par module

| Règle | Titre | Description | Seuil |
|---|---|---|---|
| **R8** | Durée de shift | Durée totale d'un shift dans un module | ≤ 11h (jour) / ≤ 10h (nuit) |
| **R9** | Nombre de tournées | Un module doit contenir au moins 2 tournées | ≥ 2 tournées |
| **R12** | Pas de chevauchement | Deux tournées d'un même module ne peuvent pas se superposer dans le temps | 0 intersection |

---

### Pauses réglementaires (R6/R7) — detail

L'application insère les pauses **automatiquement** dans le planning Gantt :

```
┌──────────┬─────────────────┬──────┬────────────────────────────┐
│ Accroch. │ Conduite ──────►│ ⏸ 45│ Suite tournée ─────────── │
│  15 min  │  max 4h30       │  min │                            │
└──────────┴─────────────────┴──────┴────────────────────────────┘
                  R6 → pause de 45 min après 4h30 de conduite
                  R7 → pause de 30 min supplémentaire après 6h de service
```

Les pauses sont **glissables** horizontalement dans le Gantt (double-clic pour réinitialiser).

---

### Contrainte lundi matin

Le **lundi matin**, les magasins n'acceptent les livraisons qu'à partir de **07h30** (contrainte liée à la fermeture du dimanche). L'optimiseur et le validateur en tiennent compte automatiquement.

---
## 11. Moteur d'optimisation automatique

Le bouton **🧠 Optimiser** déclenche un algorithme en plusieurs étapes, exécuté entièrement dans le navigateur.

### Architecture de l'algorithme

```
┌─────────────────────────────────────────────────────────────┐
│                  Phase 1 — Construction                      │
│  Pour chaque flux (trié par UT décroissant) :               │
│  ┌─────────────────────────────────────────────────────┐    │
│  │  Option A : insérer dans une tournée existante      │    │
│  │  → cherche la tournée avec le coût marginal minimal │    │
│  │  → vérifie capacité, compatibilité, créneaux, durée │    │
│  └─────────────────────────────────────────────────────┘    │
│  ┌─────────────────────────────────────────────────────┐    │
│  │  Option B : créer une nouvelle tournée dédiée       │    │
│  │  → heure de départ calculée selon le créneau cible  │    │
│  └─────────────────────────────────────────────────────┘    │
│  → Choisit la moins coûteuse entre A et B                   │
└──────────────────────────┬──────────────────────────────────┘
                           ▼
┌─────────────────────────────────────────────────────────────┐
│                  Phase 2 — LNS (500 itérations)              │
│  Destroy : retire 15% des flux aléatoirement               │
│  Repair  : réinsère les flux orphelins (comme Phase 1)      │
│  Acceptation simulée (recuit simulé, T → 0)                │
└──────────────────────────┬──────────────────────────────────┘
                           ▼
┌─────────────────────────────────────────────────────────────┐
│              Consolidation en modules                        │
│  Bin packing : affecte les tournées dans des modules        │
│  Priorité : même entrepôt > même zone > cross-zone          │
│  Contrainte : durée de shift ≤ 11h/10h                     │
│  Auto-split : coupures de shift si dépassement             │
└──────────────────────────┬──────────────────────────────────┘
                           ▼
┌─────────────────────────────────────────────────────────────┐
│         Correction anti-chevauchement (R12)                 │
│  Ajuste les heures de départ pour éviter les superpositions │
│  Valide les créneaux magasin après ajustement              │
└─────────────────────────────────────────────────────────────┘
```

### Fonction de coût

```
Coût = A (€ fixe/module) + B × km + C × heures
```

- **A** = coût fixe par module (défaut : 10 €)
- **B** = coût kilométrique (0,45 €/km sec, 0,55 €/km frais)
- **C** = coût horaire (35 €/h jour, 47 €/h nuit)

Ces paramètres sont ajustables dans le **panneau KPI** (🔍 Valider).

### Durée typique

| Volume | Durée estimation |
|---|---|
| < 50 flux | ~10 secondes |
| 50 – 150 flux | ~30 – 60 secondes |
| > 150 flux | ~1 – 2 minutes |

Une barre de progression indique l'avancement. Le bouton **Annuler** arrête l'optimisation à n'importe quel moment.

---
## 12. Tableau de bord KPI

Le panneau **🔍 Valider** contient un tableau de bord KPI automatiquement calculé.

### Indicateurs disponibles

| KPI | Description |
|---|---|
| **Coût financier / UT** | Coût total (fixe + km + heures) divisé par le nombre d'UT livrées |
| **Coût carbone / UT** | Émissions CO₂ converties en € selon le prix de la tonne carbone |
| **Taux remplissage moyen global** | UT chargées / capacité totale des remorques (sur tout le trajet) |
| **Taux remplissage moyen au départ** | UT chargées / capacité à la sortie de l'entrepôt |
| **% km à vide** | Proportion de kilomètres effectués sans marchandise (retours à vide) |

### Paramètres ajustables

| Paramètre | Libellé | Valeur par défaut |
|---|---|---|
| A | Coût fixe par module | 10 €/module |
| B sec | Coût km produits secs | 0,45 €/km |
| B frais | Coût km produits frais | 0,55 €/km |
| C jour | Coût horaire de jour | 35 €/h |
| C nuit | Coût horaire de nuit | 47 €/h |
| CO₂/km | Émissions | 900 g/km |
| Prix carbone | Prix de la tonne CO₂ | 50 €/t |

> Un **détail par module** (expandable) liste le coût de chaque module individuellement.

---
## 13. Gestion des sessions

### Sauvegarder une session

**💾 Exporter JSON** génère un fichier `.json` contenant :
- Le plan complet (affectation de tous les flux)
- Les modules et leur composition
- Les positions de chaque tournée (heures de départ)
- Les coupures de shift manuelles
- Les paramètres de chaque tournée (retour, accrochage, délais emballage…)

### Restaurer une session

**📥 Importer** charge un fichier JSON précédemment exporté.  
Le plan est immédiatement restauré dans le Gantt.

> **Conseil** : exporter une session JSON après chaque séance de travail. Le fichier JSON sert de point de reprise et peut être partagé entre planificateurs.

### Persistance locale

Le **distancier** est automatiquement mémorisé dans le `localStorage` du navigateur (même ordinateur, même navigateur). Les autres données (plan, modules) ne sont pas persistées automatiquement — pensez à exporter régulièrement.

### Export CSV

**📋 CSV** produit un tableau exportable dans Excel avec, pour chaque livraison planifiée :

| Colonne | Contenu |
|---|---|
| Module | Identifiant du module chauffeur |
| Tournée | Identifiant de la tournée |
| Entrepôt | Lieu de départ |
| Magasin | Lieu de livraison |
| Marchandise | Type de produit |
| UT | Volume livré |
| Heure d'arrivée | Heure estimée au magasin |
| Jour | Jour de livraison |

---
## 14. Astuces et raccourcis

### Lecture du Gantt

| Élément visuel | Signification |
|---|---|
| Barre **rouge** en haut de la ligne | Shift conducteur de jour (≤ 11h) |
| Barre **violette** en haut de la ligne | Shift conducteur de nuit (≤ 10h) |
| Barre **rayée rouge** | Dépassement de la durée légale de shift |
| Bloc coloré | Livraison (couleur = code zone de l'entrepôt) |
| Bloc **gris** → | Trajet entrepôt → magasin |
| Bloc **gris foncé** ← | Retour à l'entrepôt |
| Bloc **⏸ orange** | Pause réglementaire (R6 ou R7) |
| Bloc **EMB** | Délai emballage (après déchargement) |
| **✂** entre deux tournées | Coupure de shift (cliquable) |
| Fond **bleu nuit** | Plage nocturne (22h–06h) |
| Barre de composition (bas) | Visualisation UT par magasin dans la remorque |
| **⚠** sur accrochage | Entrepôt de départ différent du retour du tour précédent |

---

### Interactions dans le Gantt

| Interaction | Effet |
|---|---|
| Clic sur un **bloc de livraison** | Affiche le détail (tooltip) |
| Clic sur le **✕** d'un bloc | Retire ce flux de la tournée |
| Glisser un **bloc de livraison** | Réordonne les livraisons dans la tournée |
| Glisser la **barre de tournée** horizontalement | Modifie l'heure de départ |
| Glisser la **barre de tournée** verticalement | Déplace la tournée dans un autre module |
| Glisser le **bloc ⏸** (pause) vers la gauche | Avance la pause dans le temps |
| Double-clic sur **⏸** | Réinitialise la position de la pause |
| Clic sur la **zone ✂** entre deux tournées | Insère/retire une coupure de shift |
| Clic sur l'**icône 🕐** d'un magasin (panneau flux) | Affiche les créneaux de réception |
| Clic sur un **module** (Gantt) | Affiche ses tournées dans le panneau droit |

---

### Bonnes pratiques

1. **Charger le distancier en premier** : sans distances, le Gantt ne peut pas calculer les durées de trajet et l'optimisation sera inefficace.

2. **Vérifier l'alerte orange** : si des paires de lieux sont absentes du distancier, elles apparaissent dans l'alerte en haut du Gantt. Compléter le distancier avant d'optimiser.

3. **Utiliser M0 comme brouillon** : le Module M0 (fond jaune) est un espace de travail temporaire. Les tournées y sont construites sans contrainte de chauffeur. Déplacez-les ensuite dans des modules numérotés.

4. **Filtrer avant d'optimiser** : si vous voulez optimiser uniquement les flux frais, activez le filtre `FL + PF` avant de lancer 🧠 Optimiser.

5. **Exporter régulièrement** : le plan n'est pas sauvegardé automatiquement. Utilisez 💾 Exporter JSON après chaque session de travail.

6. **Valider avant d'exporter** : cliquez sur 🔍 Valider → Tout valider pour identifier les non-conformités avant de transmettre le CSV.

7. **Attention au lundi matin** : les flux du lundi ne peuvent pas être livrés avant 07h30. L'optimiseur en tient compte, mais une modification manuelle peut créer une infraction — vérifiez avec R10.

---

### FAQ rapide

**Q : Un flux n'apparaît pas dans le panneau gauche — pourquoi ?**  
→ Vérifiez les filtres actifs (Marchandise, Magasin, Entrepôt, Zone). Vérifiez également que le bon jour est sélectionné.

**Q : La barre de shift dépasse la durée légale — que faire ?**  
→ Insérer une coupure ✂ entre deux tournées du module, ou déplacer une tournée dans un autre module.

**Q : L'optimiseur n'a pas affecté certains flux — pourquoi ?**  
→ Deux causes principales : (1) le distancier ne couvre pas tous les trajets nécessaires, (2) aucun créneau compatible n'existe pour ces flux. Vérifiez la cartographie réception.

**Q : Deux tournées d'un module se chevauchent (R12) — comment corriger ?**  
→ Déplacer manuellement l'une des tournées (glisser) ou relancer 🧠 Optimiser (la phase de correction anti-chevauchement recalcule les offsets).

**Q : Le panneau droit n'affiche pas les tournées du bon jour ?**  
→ Le panneau filtre automatiquement par le jour actif. Si vous avez changé de jour, cliquez à nouveau sur le module.

---

*Document généré pour l'outil de planification Transport v7 — Auchan Région Nord.*